In [2]:
import ee
ee.Initialize(project="sentinel-487715")

# Ghana boundary
ghana = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Ghana")) \
    .geometry()

# Roads asset
roads = ee.FeatureCollection("projects/sentinel-487715/assets/ghana_roads")

# Quarter ranges
quarters = {
    "Q1": ("01-01","03-31"),
    "Q2": ("04-01","06-30"),
    "Q3": ("07-01","09-30"),
    "Q4": ("10-01","12-31"),
}

def make_img(start, end):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ghana)
          .filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .select(["B2","B3","B4","B8","B11","B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8","B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11","B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3","B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

# Launch exports
tasks = []
for year in range(2020, 2024):
    for q, (start_mmdd, end_mmdd) in quarters.items():
        start = f"{year}-{start_mmdd}"
        end = f"{year}-{end_mmdd}"

        img = make_img(start, end)
        reducer = ee.Reducer.mean().forEachBand(img)

        stats_fc = img.reduceRegions(
            collection=roads,
            reducer=reducer,
            scale=20,
            tileScale=16
        ).map(lambda f: f.set({"year": year, "quarter": q}))

        desc = f"ghana_roads_{year}_{q}"
        task = ee.batch.Export.table.toDrive(
            collection=stats_fc,
            description=desc,
            fileFormat="CSV",
            folder="GEE_Exports"
        )
        task.start()
        tasks.append(task)

print(f"Started {len(tasks)} export tasks.")


Started 16 export tasks.


In [1]:
# Export with cloud threshold of 40% for Q3 (2020–2023)

import ee
ee.Initialize(project="sentinel-487715")

# Ghana boundary
ghana = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Ghana")) \
    .geometry()

# Roads asset
roads = ee.FeatureCollection("projects/sentinel-487715/assets/ghana_roads")

def make_img_q3(start, end, cloud_thresh=40):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ghana)
          .filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_thresh))
          .select(["B2","B3","B4","B8","B11","B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8","B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11","B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3","B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

tasks = []
for year in range(2020, 2024):
    start = f"{year}-07-01"
    end   = f"{year}-09-30"

    img = make_img_q3(start, end, cloud_thresh=40)
    reducer = ee.Reducer.mean().forEachBand(img)

    stats_fc = img.reduceRegions(
        collection=roads,
        reducer=reducer,
        scale=20,
        tileScale=16
    ).map(lambda f: f.set({"year": year, "quarter": "Q3", "cloud_thresh": 40}))

    desc = f"ghana_roads_{year}_Q3_cloud40"
    task = ee.batch.Export.table.toDrive(
        collection=stats_fc,
        description=desc,
        fileNamePrefix=desc,
        folder="GEE_Exports",
        fileFormat="CSV"
    )
    task.start()
    tasks.append(task)
    print("Started:", desc, "| task id:", task.id)

print(f"Started {len(tasks)} Q3 export tasks (2020–2023, cloud < 40%).")


Started: ghana_roads_2020_Q3_cloud40 | task id: BDE6G6FRR3QMH5ZKG7IBX4PK
Started: ghana_roads_2021_Q3_cloud40 | task id: SJOVWGBJSIFVQXUWCFCDK7T2
Started: ghana_roads_2022_Q3_cloud40 | task id: 4QBVTTRHS5J3FELNF4QNIOFO
Started: ghana_roads_2023_Q3_cloud40 | task id: BEAGX7HZORMG2XZAGXZQTY5D
Started 4 Q3 export tasks (2020–2023, cloud < 40%).


In [9]:
import ee
ee.Initialize(project="sentinel-487715")

# -----------------------------
# Asset: RoadCondition only
# -----------------------------
RC = ee.FeatureCollection("projects/sentinel-487715/assets/nigeria/RoadCondition")

EXPORT_FOLDER = "GEE_Exports"
CLOUD_MAX = 20
SCALE_M = 20
TILE_SCALE = 16
N_CHUNKS = 1   # keep chunking because the road geometry set itself may still be large

# -----------------------------
# Normalize ROADCODE
# -----------------------------
def with_code(fc):
    def _f(f):
        c = ee.String(ee.Algorithms.If(f.get("ROADCODE"), f.get("ROADCODE"), ""))
        c = c.trim().replace("\\s+", "")
        return f.set("ROADCODE_N", c)
    return fc.map(_f)

rc = with_code(RC).randomColumn("chunk_rand", 42)
aoi = rc.geometry().bounds()

# -----------------------------
# Sentinel image builder
# -----------------------------
def make_img(start_date, end_date):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(aoi)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
          .select(["B2", "B3", "B4", "B8", "B11", "B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8", "B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(
               s2.select("B11").add(s2.select("B4"))
               .add(s2.select("B8")).add(s2.select("B2"))
           )
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

def chunk_filter(i, n_chunks):
    lo = i / n_chunks
    hi = (i + 1) / n_chunks
    return ee.Filter.And(
        ee.Filter.gte("chunk_rand", lo),
        ee.Filter.lt("chunk_rand", hi)
    )

# -----------------------------
# 2020 quarterly exports
# -----------------------------
quarters = {
    "Q1": ("2020-01-01", "2020-03-31"),
    "Q2": ("2020-04-01", "2020-06-30"),
    "Q3": ("2020-07-01", "2020-09-30"),
    "Q4": ("2020-10-01", "2020-12-31"),
}

tasks = []

for q, (start, end) in quarters.items():
    img = make_img(start, end)
    reducer = ee.Reducer.mean().forEachBand(img)

    for i in range(N_CHUNKS):
        rc_chunk = rc.filter(chunk_filter(i, N_CHUNKS))

        stats_fc = img.reduceRegions(
            collection=rc_chunk,
            reducer=reducer,
            scale=SCALE_M,
            tileScale=TILE_SCALE
        ).map(lambda f: f.set({
            "year": 2020,
            "quarter": q,
            "chunk": i + 1
        }))

        desc = f"nigeria_rc_s2_2020_{q}_part{i+1}"
        task = ee.batch.Export.table.toDrive(
            collection=stats_fc,
            description=desc,
            folder=EXPORT_FOLDER,
            fileNamePrefix=desc,
            fileFormat="CSV"
        )
        task.start()
        tasks.append(task)
        print("Started:", desc, task.id)

print("Total tasks started:", len(tasks))


Started: nigeria_rc_s2_2020_Q1_part1 X3QZUU27HBB5JXLXLC5ZEEJL
Started: nigeria_rc_s2_2020_Q2_part1 MGBMWVM6URWIW2M56CBWPAAQ
Started: nigeria_rc_s2_2020_Q3_part1 MDK52IWAK7MVGGV6IK5HJRC3
Started: nigeria_rc_s2_2020_Q4_part1 7PHZPOAHMOIASEHFR5SDVII6
Total tasks started: 4
